<a href="https://colab.research.google.com/github/Shreya-1910/ArtificalMethodsCW/blob/main/HybridFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import time
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Install pyswarms for PSO
!pip install pyswarms --quiet
import pyswarms as ps

print("All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 1.8 MB/s eta 0:00:00
All libraries imported successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Install pyswarms for PSO
!pip install pyswarms --quiet
import pyswarms as ps

print("All libraries imported and Drive mounted successfully!")

# Define paths
sample_data_path = '/content/drive/MyDrive/AIMfiles/PreprocessedData/SampleData'
full_data_path = '/content/drive/MyDrive/AIMfiles/PreprocessedData'  # Full data is in parent folder

# Load SAMPLE data (for optimization)
X_sample_train = np.load(os.path.join(sample_data_path, 'X_sample_train.npy'))
X_sample_val = np.load(os.path.join(sample_data_path, 'X_sample_val.npy'))
y_sample_train = np.load(os.path.join(sample_data_path, 'y_sample_train.npy'))
y_sample_val = np.load(os.path.join(sample_data_path, 'y_sample_val.npy'))

print(f"Sample data loaded successfully:")
print(f"  X_sample_train shape: {X_sample_train.shape}")
print(f"  y_sample_train shape: {y_sample_train.shape}")
print(f"  X_sample_val shape:   {X_sample_val.shape}")
print(f"  y_sample_val shape:   {y_sample_val.shape}")
print(f"  Number of features:   {X_sample_train.shape[1]}")
print(f"\n  Train attack ratio: {y_sample_train.mean():.2%}")
print(f"  Val attack ratio:   {y_sample_val.mean():.2%}")

# Assign sample data to optimization variables
X_opt_train = X_sample_train
y_opt_train = y_sample_train
X_opt_val = X_sample_val
y_opt_val = y_sample_val

# Define n_features
n_features = X_sample_train.shape[1]
feat_names_list = [f'feature_{i}' for i in range(n_features)]

# Load FULL dataset (for final training and testing)
print("\nLoading full dataset...")
X_test = np.load(os.path.join(full_data_path, 'X_test.npy'))
y_test = np.load(os.path.join(full_data_path, 'y_test.npy'))
X_tr = np.load(os.path.join(full_data_path, 'X_tr.npy'))
y_tr = np.load(os.path.join(full_data_path, 'y_tr.npy'))
X_val = np.load(os.path.join(full_data_path, 'X_val.npy'))
y_val = np.load(os.path.join(full_data_path, 'y_val.npy'))

print(f"Full dataset loaded:")
print(f"  X_tr shape: {X_tr.shape}")
print(f"  y_tr shape: {y_tr.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  y_val shape: {y_val.shape}")
print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")

# Scale the full training and test data for the final model
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_test_scaled = scaler.transform(X_test)

print(f"\nData scaling complete:")
print(f"  X_tr_scaled shape: {X_tr_scaled.shape}")
print(f"  X_test_scaled shape: {X_test_scaled.shape}")
print(f"  Training attack ratio: {y_tr.mean():.2%}")
print(f"  Test attack ratio: {y_test.mean():.2%}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All libraries imported and Drive mounted successfully!
Sample data loaded successfully:
  X_sample_train shape: (70626, 70)
  y_sample_train shape: (70626,)
  X_sample_val shape:   (17657, 70)
  y_sample_val shape:   (17657,)
  Number of features:   70

  Train attack ratio: 16.88%
  Val attack ratio:   16.88%

Loading full dataset...
Full dataset loaded:
  X_tr shape: (1765652, 70)
  y_tr shape: (1765652,)
  X_val shape: (252237, 70)
  y_val shape: (252237,)
  X_test shape: (504473, 70)
  y_test shape: (504473,)

Data scaling complete:
  X_tr_scaled shape: (1765652, 70)
  X_test_scaled shape: (504473, 70)
  Training attack ratio: 16.88%
  Test attack ratio: 16.88%


In [19]:
# PSO-GA Hybrid Parameters
GA_POP = 30
GA_GENS = 40
GA_MUTATION = 0.05
GA_ELITES = 2

# Random Forest Parameters for optimization
RF_N_ESTIMATORS = 20
RF_MAX_DEPTH = 10
RF_RANDOM_STATE = 42
RF_N_JOBS = -1
RF_CLASS_WEIGHT = 'balanced'

print("Hybrid PSO-GA Parameters:")
print(f"  Population Size: {GA_POP}")
print(f"  Generations: {GA_GENS}")
print(f"  Mutation Rate: {GA_MUTATION}")
print(f"  Elite Count: {GA_ELITES}")
print(f"\nRandom Forest Parameters:")
print(f"  n_estimators: {RF_N_ESTIMATORS}")
print(f"  max_depth: {RF_MAX_DEPTH}")
print(f"  class_weight: {RF_CLASS_WEIGHT}")

def fitness_function(individuals, mode='pso'):
    """
 fitness function for hybrid PSO-GA
    """
    scores = []

    for ind in individuals:
        # Convert to binary mask
        if mode == 'pso':
            mask = ind > 0.5
        else:  # GA
            mask = ind.astype(bool)

        # Penalty for empty mask
        if mask.sum() == 0:
            scores.append(1e9)
            continue

        # Train Random Forest with selected features
        rf = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            random_state=RF_RANDOM_STATE,
            n_jobs=RF_N_JOBS,
            class_weight=RF_CLASS_WEIGHT
        )

        rf.fit(X_opt_train[:, mask], y_opt_train)
        y_pred = rf.predict(X_opt_val[:, mask])
        f1 = f1_score(y_opt_val, y_pred, average='weighted', zero_division=0)

        # NO PENALTY - just minimize (1 - F1)
        scores.append(1 - f1)

    return np.array(scores)

print("\nFitness function defined successfully!")

Hybrid PSO-GA Parameters:
  Population Size: 30
  Generations: 40
  Mutation Rate: 0.05
  Elite Count: 2

Random Forest Parameters:
  n_estimators: 20
  max_depth: 10
  class_weight: balanced

Fitness function defined successfully!


In [20]:
print("\n" + "="*60)
print("PHASE 1: PSO OPTIMIZATION")
print("="*60)

pso_options = {'c1': 1.5, 'c2': 1.5, 'w': 0.7}
bounds = (np.zeros(n_features), np.ones(n_features))

optimizer = ps.single.GlobalBestPSO(
    n_particles=GA_POP,
    dimensions=n_features,
    options=pso_options,
    bounds=bounds
)

start_pso = time.time()
best_cost_pso, best_pos_pso = optimizer.optimize(
    lambda x: fitness_function(x, mode='pso'),
    iters=GA_GENS,
    verbose=True
)
pso_time = time.time() - start_pso

# Convert PSO best position to binary mask
pso_best_mask = best_pos_pso > 0.5
pso_best_f1 = 1 - best_cost_pso
print(f"\n PSO Phase completed in {pso_time/60:.2f} minutes")
print(f"  Best features selected: {pso_best_mask.sum()}/{n_features}")
print(f"  Best F1 Score: {pso_best_f1:.4f}")

2026-04-19 10:28:27,910 - pyswarms.single.global_best - INFO - Optimize for 40 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.7}



PHASE 1: PSO OPTIMIZATION


pyswarms.single.global_best: 100%|██████████|40/40, best_cost=0.00187
2026-04-19 11:17:02,440 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0018705755148520353, best pos: [0.70278157 0.38065182 0.97580527 0.77195847 0.63933437 0.37125856
 0.74860216 0.03163848 0.66050881 0.746137   0.6588409  0.09973129
 0.45141229 0.62901796 0.48049204 0.54540956 0.28023948 0.39675573
 0.65271643 0.49386334 0.48894661 0.11623976 0.70902529 0.05362473
 0.68122596 0.50135885 0.04430558 0.38963537 0.48846544 0.6897797
 0.40600386 0.50618075 0.93028981 0.02121354 0.18895694 0.53315817
 0.67469896 0.28875352 0.28422197 0.42332364 0.23766577 0.44294097
 0.68141037 0.72085248 0.41049542 0.61678668 0.90962968 0.19281555
 0.06141874 0.10595602 0.06793102 0.59008546 0.10020818 0.50788162
 0.77935155 0.40963806 0.67639887 0.28109919 0.79435072 0.0777958
 0.27882794 0.27731652 0.14583186 0.49574506 0.41495204 0.28811612
 0.23148059 0.26007713 0.55379573 0.2293893 ]



 PSO Phase completed in 48.58 minutes
  Best features selected: 29/70
  Best F1 Score: 0.9981


In [21]:
def initialize_population_from_pso(pop_size, n_feat, pso_solution):
    """Initialize GA population using PSO solution"""
    population = []

    # Add PSO solution as elite individual
    population.append(pso_solution.astype(int).copy())

    # Generate remaining population with some bias towards PSO solution
    for i in range(pop_size - 1):
        # Create individual with high probability of inheriting from PSO
        individual = pso_solution.copy()
        # Randomly flip some bits based on mutation rate
        mutation_mask = np.random.random(n_feat) < GA_MUTATION
        individual[mutation_mask] = 1 - individual[mutation_mask]
        population.append(individual.astype(int))

    return np.array(population)

def tournament_selection(population, fitness_scores, tournament_size=3):
    """Tournament selection for GA"""
    selected = []
    for _ in range(len(population)):
        tournament_idx = np.random.choice(len(population), tournament_size, replace=False)
        winner_idx = tournament_idx[np.argmin(fitness_scores[tournament_idx])]
        selected.append(population[winner_idx].copy())
    return np.array(selected)

def crossover(parent1, parent2, crossover_rate=0.8):
    """Single-point crossover for GA"""
    if np.random.random() < crossover_rate:
        point = np.random.randint(1, len(parent1))
        child1 = np.concatenate([parent1[:point], parent2[point:]])
        child2 = np.concatenate([parent2[:point], parent1[point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()

def mutate(individual, mutation_rate=GA_MUTATION):
    """Bit-flip mutation for GA"""
    for i in range(len(individual)):
        if np.random.random() < mutation_rate:
            individual[i] = 1 - individual[i]
    return individual

print(" GA functions defined successfully!")

 GA functions defined successfully!


In [22]:
print("\n" + "="*60)
print("PHASE 2: GA OPTIMIZATION (Initialized with PSO)")
print("="*60)

# Initialize population using PSO solution
population = initialize_population_from_pso(GA_POP, n_features, pso_best_mask)
best_fitness_history = []
avg_fitness_history = []
feature_count_history = []

start_ga = time.time()

for generation in range(GA_GENS):
    # Evaluate fitness
    fitness_scores = fitness_function(population, mode='ga')

    # Track best and average
    best_idx = np.argmin(fitness_scores)
    best_fitness_history.append(fitness_scores[best_idx])
    avg_fitness_history.append(np.mean(fitness_scores))
    feature_count_history.append(population[best_idx].sum())

    # Store elite individuals
    elite_indices = np.argsort(fitness_scores)[:GA_ELITES]
    elites = population[elite_indices].copy()

    # Selection
    selected_population = tournament_selection(population, fitness_scores)

    # Crossover
    next_population = []
    for i in range(0, len(selected_population), 2):
        if i+1 < len(selected_population):
            child1, child2 = crossover(selected_population[i], selected_population[i+1])
            next_population.extend([child1, child2])
        else:
            next_population.append(selected_population[i])

    # Mutation
    next_population = [mutate(ind) for ind in next_population]

    # Elitism
    next_population[:GA_ELITES] = elites

    population = np.array(next_population)

    # Print progress every 10 generations
    if (generation + 1) % 10 == 0:
        best_f1 = 1 - best_fitness_history[-1]
        print(f"Generation {generation+1}/{GA_GENS}: Best Fitness = {best_fitness_history[-1]:.4f}, "
              f"Best F1 = {best_f1:.4f}, Features = {population[best_idx].sum()}")

ga_time = time.time() - start_ga

# Get best solution from hybrid model
final_fitness = fitness_function(population, mode='ga')
best_idx = np.argmin(final_fitness)
hybrid_best_mask = population[best_idx].astype(bool)
hybrid_best_f1 = 1 - final_fitness[best_idx]

print(f"\n Hybrid PSO-GA Phase completed in {ga_time/60:.2f} minutes")
print(f"  Best features selected: {hybrid_best_mask.sum()}/{n_features}")
print(f"  Best F1 Score: {hybrid_best_f1:.4f}")
print(f"\n Improvement from PSO: {(hybrid_best_f1 - pso_best_f1)*100:.2f}%")


PHASE 2: GA OPTIMIZATION (Initialized with PSO)
Generation 10/40: Best Fitness = 0.0016, Best F1 = 0.9984, Features = 29
Generation 20/40: Best Fitness = 0.0016, Best F1 = 0.9984, Features = 26
Generation 30/40: Best Fitness = 0.0016, Best F1 = 0.9984, Features = 26
Generation 40/40: Best Fitness = 0.0016, Best F1 = 0.9984, Features = 26

 Hybrid PSO-GA Phase completed in 41.51 minutes
  Best features selected: 26/70
  Best F1 Score: 0.9984

 Improvement from PSO: 0.03%


In [23]:
print("\n" + "="*60)
print("FINAL MODEL TRAINING ON FULL DATASET")
print("="*60)

# Train final Random Forest on full training data with selected features
final_rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RF_RANDOM_STATE,
    n_jobs=RF_N_JOBS,
    class_weight=RF_CLASS_WEIGHT
)

# Train on full scaled training data with selected features
final_rf.fit(X_tr_scaled[:, hybrid_best_mask], y_tr)

print(f" Final model trained on full dataset")
print(f"  Training samples: {X_tr_scaled.shape[0]}")
print(f"  Selected features: {hybrid_best_mask.sum()}")
print(f"  Training complete!")


FINAL MODEL TRAINING ON FULL DATASET
 Final model trained on full dataset
  Training samples: 1765652
  Selected features: 26
  Training complete!


In [28]:
print("\n" + "="*60)
print("Model evaluvation on test set")
print("="*60)

# Predict on test set
y_test_pred = final_rf.predict(X_test_scaled[:, hybrid_best_mask])
y_test_pred_proba = final_rf.predict_proba(X_test_scaled[:, hybrid_best_mask])

# Calculate all metrics
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

# Calculate ROC-AUC
if len(np.unique(y_test)) == 2:
    roc_auc = roc_auc_score(y_test, y_test_pred_proba[:, 1])
else:
    roc_auc = roc_auc_score(y_test, y_test_pred_proba, multi_class='ovr', average='weighted')

# Print metrics
print(f"\n{'Metric':<15} {'Score':<10}")
print("-" * 30)
print(f"{'Accuracy':<15} {accuracy:.4f}")
print(f"{'Precision':<15} {precision:.4f}")
print(f"{'Recall':<15} {recall:.4f}")
print(f"{'F1-Score':<15} {f1:.4f}")
print(f"{'ROC-AUC':<15} {roc_auc:.4f}")
print(f"{'Features Used':<15} {hybrid_best_mask.sum()}/{n_features}")

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_test_pred)

if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    print(f"\nConfusion Matrix:")
    print(f"  TN: {tn:,}  |  FP: {fp:,}")
    print(f"  FN: {fn:,}  |  TP: {tp:,}")
    print(f"\n  False Alarm Rate: {fp/(fp+tn)*100:.2f}%")
    print(f"  Missed Attack Rate: {fn/(fn+tp)*100:.2f}%")


Model evaluvation on test set

Metric          Score     
------------------------------
Accuracy        0.9980
Precision       0.9980
Recall          0.9980
F1-Score        0.9980
ROC-AUC         0.9999
Features Used   26/70

Confusion Matrix:
  TN: 418,550  |  FP: 747
  FN: 267  |  TP: 84,909

  False Alarm Rate: 0.18%
  Missed Attack Rate: 0.31%
